# Comparacion de Baselines — MOPPO vs. alternativas

Compara 6 metodos sobre el mismo grafo (ego-Facebook n=100) con las mismas metricas.

| Metodo | Tipo | Training runs |
|--------|------|---------------|
| **MOPPO** | RL multi-objetivo, alpha ~ U(0,1) | 1 |
| **Fixed-alpha PPO (K=5)** | RL mono-objetivo x5 | 5 |
| **NSGA-II** | Evolutivo, busqueda directa en grafo | 0 |
| **MOEA/D** | Evolutivo, descomposicion Tchebycheff | 0 |
| **Greedy Liu & Terzi** | Heuristico deterministico | 0 |
| **Politica aleatoria** | Cota inferior | 0 |

**Metricas:** hypervolume, Pareto overlay, fairness audit comparativo (disparity index).

In [ ]:
import sys
import subprocess
import os
import warnings
from pathlib import Path
from itertools import combinations

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.optim as optim

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.float_format', '{:.5f}'.format)

# pymoo para NSGA-II
try:
    import pymoo
except ImportError:
    print('Instalando pymoo...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pymoo', '-q'], check=True)
    import pymoo

# Resolucion de rutas
cwd = Path.cwd()
if (cwd / 'scripts' / 'python').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'scripts' / 'python').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError('Ejecutar desde project root o notebooks/')

SCRIPTS_DIR = str(PROJECT_ROOT / 'scripts' / 'python')
DATA_DIR    = str(PROJECT_ROOT / 'data')
OUT_DIR     = str(PROJECT_ROOT / 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, SCRIPTS_DIR)

print(f'Project root : {PROJECT_ROOT}')
print(f'PyTorch      : {torch.__version__}')
print(f'pymoo        : {pymoo.__version__}')

## 1. Setup compartido

Carga el grafo, importa modulos y define funciones de evaluacion compartidas por todos los metodos.

In [ ]:
from graph_anon_morl.datasets   import load_facebook_ego
from graph_anon_morl.env        import GraphAnonEnv
from graph_anon_morl.models     import MOPPOActorCritic
from graph_anon_morl.utils      import compute_k_anonymity_reward, compute_utility_reward
from graph_anon_morl.evaluation import compute_pareto_front, hypervolume_indicator
from graph_anon_morl.audit      import degree_quartile_audit, fairness_disparity_index
from train_main                 import collect_rollout, compute_gae, ppo_update

G_fb      = load_facebook_ego(n_nodes=100, seed=42, data_dir=DATA_DIR)
K_ANON    = 2
T_STEPS   = 50
LR        = 3e-4
N_STEPS   = 256
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ALL_EDGES = list(combinations(range(G_fb.number_of_nodes()), 2))

def make_env():
    return GraphAnonEnv(G_fb, k=K_ANON, T=T_STEPS)

_ref   = make_env()
obs_dim   = _ref.observation_space.shape[0]
n_actions = _ref.n_actions

print(f'Grafo       : ego-Facebook n={G_fb.number_of_nodes()}, m={G_fb.number_of_edges()}')
print(f'Action space: {n_actions}')
print(f'obs_dim     : {obs_dim}')
print(f'Device      : {device}')

# ---- Funciones compartidas ----

def evaluate_at_alpha(env, model, alpha, n_episodes=5):
    model.eval()
    ep_rewards = []
    with torch.no_grad():
        for _ in range(n_episodes):
            obs, _ = env.reset(weight=[alpha, 1.0 - alpha])
            done = False
            total_r = np.zeros(2)
            while not done:
                obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
                action, _, _, _ = model.get_action(obs_t, deterministic=True)
                obs, _, terminated, truncated, info = env.step(action.item())
                total_r += info['reward_vec']
                done = terminated or truncated
            ep_rewards.append(total_r)
    mean_r = np.mean(ep_rewards, axis=0)
    return float(alpha), float(mean_r[0]), float(mean_r[1])

def evaluate_sweep(env, model, n_weights=11, n_episodes=5):
    return [evaluate_at_alpha(env, model, float(a), n_episodes)
            for a in np.linspace(0, 1, n_weights)]

def get_final_graph(env, model, alpha=0.5):
    obs, _ = env.reset(weight=[alpha, 1.0 - alpha])
    done = False
    with torch.no_grad():
        while not done:
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            action, _, _, _ = model.get_action(obs_t, deterministic=True)
            obs, _, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
    return env.G.copy(), env.G0

def compute_hv(results):
    pts = [(r[1], r[2]) for r in results]
    idx = compute_pareto_front(pts)
    return hypervolume_indicator([pts[i] for i in idx])

def apply_flips(G0, flip_mask):
    G = G0.copy()
    for i, flip in enumerate(flip_mask):
        if flip:
            u, v = ALL_EDGES[i]
            if G.has_edge(u, v):
                G.remove_edge(u, v)
            else:
                G.add_edge(u, v)
    return G

print('Setup completo.')

## 2. MOPPO — entrenamiento y evaluacion

Alpha aleatorio por episodio: un solo entrenamiento cubre toda la frontera de Pareto.

> Para el paper usar `N_ITER_MOPPO=500`. Aqui usamos 50 para demo.

In [ ]:
N_ITER_MOPPO = 50
torch.manual_seed(42)
np.random.seed(42)

model_moppo = MOPPOActorCritic(obs_dim, n_actions, hidden_dim=128).to(device)
opt_moppo   = optim.Adam(model_moppo.parameters(), lr=LR)
env_moppo   = make_env()

print(f'Entrenando MOPPO — {N_ITER_MOPPO} iteraciones, alpha ~ U(0,1) por episodio')
for i in range(N_ITER_MOPPO):
    obs_b, acts_b, logps_b, vals_b, rews_b, dones_b = collect_rollout(
        env_moppo, model_moppo, N_STEPS, device
    )
    advs, rets = compute_gae(rews_b, vals_b, dones_b)
    loss = ppo_update(model_moppo, opt_moppo, obs_b, acts_b, logps_b, rets, advs)
    if (i + 1) % 10 == 0:
        print(f'  [{i+1:3d}/{N_ITER_MOPPO}]  loss={loss:.5f}  mean_r={rews_b.mean():.5f}')

print('Listo.')

In [ ]:
results_moppo = evaluate_sweep(make_env(), model_moppo, n_weights=11, n_episodes=5)
hv_moppo      = compute_hv(results_moppo)
print(f'MOPPO  HV={hv_moppo:.5f}  (1 training run)')
for alpha, rp, ru in results_moppo:
    print(f'  alpha={alpha:.2f}  r_priv={rp:.5f}  r_util={ru:.5f}')

## 3. Baseline: Politica Aleatoria

Cota inferior. Flips uniformemente aleatorios sin ninguna estrategia.

In [ ]:
def evaluate_random(n_weights=11, n_episodes=5):
    env_r = make_env()
    results = []
    for alpha in np.linspace(0, 1, n_weights):
        ep_rewards = []
        for _ in range(n_episodes):
            obs, _ = env_r.reset(weight=[float(alpha), float(1.0 - alpha)])
            total_r = np.zeros(2)
            done = False
            while not done:
                obs, _, terminated, truncated, info = env_r.step(env_r.action_space.sample())
                total_r += info['reward_vec']
                done = terminated or truncated
            ep_rewards.append(total_r)
        mean_r = np.mean(ep_rewards, axis=0)
        results.append((float(alpha), float(mean_r[0]), float(mean_r[1])))
    return results

results_random = evaluate_random()
hv_random      = compute_hv(results_random)
print(f'Aleatorio  HV={hv_random:.5f}')
print(f'  r_priv promedio: {np.mean([r[1] for r in results_random]):.5f}')
print(f'  r_util promedio: {np.mean([r[2] for r in results_random]):.5f}')

## 4. Baseline: Greedy Liu & Terzi (2008)

Algoritmo deterministico: agrega aristas a nodos en clases singleton hasta lograr k-anonimato. Produce **un solo punto** (privacidad maxima, utilidad ignorada). Referencia: Liu & Terzi, KDD 2008 [4].

In [ ]:
def greedy_k_anonymize(G0, k=2, max_ops=500):
    G = G0.copy()
    for _ in range(max_ops):
        deg_counts = {}
        for _, d in G.degree():
            deg_counts[d] = deg_counts.get(d, 0) + 1
        at_risk = [nd for nd, d in G.degree() if deg_counts[d] < k]
        if not at_risk:
            break
        u     = at_risk[0]
        u_deg = G.degree(u)
        # Priorizar: conectar con nodo de mismo grado (funde clases de equivalencia)
        same = [v for v, d in G.degree()
                if d == u_deg and v != u and not G.has_edge(u, v)]
        if same:
            G.add_edge(u, same[0])
        else:
            other = [v for v in G.nodes() if v != u and not G.has_edge(u, v)]
            if not other:
                break
            G.add_edge(u, other[0])
    return G

G_greedy   = greedy_k_anonymize(G_fb, k=K_ANON)
rp_greedy  = compute_k_anonymity_reward(G_greedy, K_ANON)
ru_greedy  = compute_utility_reward(G_greedy, G_fb)
hv_greedy  = hypervolume_indicator([(rp_greedy, ru_greedy)])

deg_counts_g = {}
for _, d in G_greedy.degree():
    deg_counts_g[d] = deg_counts_g.get(d, 0) + 1
min_class = min(deg_counts_g.values())

print('Greedy Liu & Terzi (2008):')
print(f'  Aristas orig / anon : {G_fb.number_of_edges()} / {G_greedy.number_of_edges()}')
print(f'  Aristas agregadas   : {G_greedy.number_of_edges() - G_fb.number_of_edges()}')
print(f'  Min clase equiv     : {min_class}  (k-anonimo: {min_class >= K_ANON})')
print(f'  r_priv              : {rp_greedy:.5f}')
print(f'  r_util              : {ru_greedy:.5f}')
print(f'  Hypervolume         : {hv_greedy:.5f}')

## 5. Baseline: Fixed-alpha PPO (K=5)

5 agentes PPO independientes con alpha fijo (0.0, 0.25, 0.5, 0.75, 1.0). Cada uno produce un solo punto en el espacio objetivo. Requiere K veces mas computo que MOPPO para cubrir K puntos.

Comparacion directa: misma arquitectura, mismos hiperparametros, unica diferencia es si alpha es fijo o aleatorio.

In [ ]:
def collect_rollout_fixed(env, model, n_steps, device, alpha):
    obs_buf, act_buf, logp_buf, val_buf, rew_buf, done_buf = [], [], [], [], [], []
    obs, _ = env.reset(weight=[alpha, 1.0 - alpha])
    for _ in range(n_steps):
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        with torch.no_grad():
            action, logp, _, value = model.get_action(obs_t)
        next_obs, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        obs_buf.append(obs)
        act_buf.append(action.item())
        logp_buf.append(logp.item())
        val_buf.append(value.item())
        rew_buf.append(reward)
        done_buf.append(float(done))
        obs = env.reset(weight=[alpha, 1.0 - alpha])[0] if done else next_obs
    return (
        torch.FloatTensor(np.array(obs_buf)).to(device),
        torch.LongTensor(act_buf).to(device),
        torch.FloatTensor(logp_buf).to(device),
        torch.FloatTensor(val_buf).to(device),
        torch.FloatTensor(rew_buf).to(device),
        torch.FloatTensor(done_buf).to(device),
    )

FIXED_ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
N_ITER_FIXED = 30
fixed_models = {}

print(f'Entrenando {len(FIXED_ALPHAS)} agentes Fixed-alpha — {N_ITER_FIXED} iters cada uno')
for alpha in FIXED_ALPHAS:
    torch.manual_seed(42)
    m = MOPPOActorCritic(obs_dim, n_actions, hidden_dim=128).to(device)
    o = optim.Adam(m.parameters(), lr=LR)
    e = make_env()
    for _ in range(N_ITER_FIXED):
        obs_b, acts_b, logps_b, vals_b, rews_b, dones_b = collect_rollout_fixed(
            e, m, N_STEPS, device, alpha
        )
        advs, rets = compute_gae(rews_b, vals_b, dones_b)
        ppo_update(m, o, obs_b, acts_b, logps_b, rets, advs)
    fixed_models[alpha] = m
    print(f'  alpha={alpha:.2f} — listo.')
print('Todos los Fixed-alpha entrenados.')

In [ ]:
results_fixed_dict = {}
for alpha, m in fixed_models.items():
    res = evaluate_at_alpha(make_env(), m, alpha, n_episodes=5)
    results_fixed_dict[alpha] = res
    print(f'  Fixed alpha={alpha:.2f}  r_priv={res[1]:.5f}  r_util={res[2]:.5f}')

results_fixed_all = list(results_fixed_dict.values())
hv_fixed          = compute_hv(results_fixed_all)
print(f'\nFixed-alpha K={len(FIXED_ALPHAS)}  HV={hv_fixed:.5f}  ({len(FIXED_ALPHAS)} training runs)')

## 6. Baseline: NSGA-II (pymoo)

Busqueda evolutiva directamente sobre el espacio de grafos finales. Cada individuo = vector binario que indica que aristas flipear desde G0. No aprende una politica — busca el grafo optimo directamente.

**Ventaja conceptual:** sin restriccion de T pasos, busca directamente el grafo final optimo.  
**Desventaja:** debe re-ejecutarse para cada nuevo alpha; no generaliza a grafos nuevos.

In [ ]:
from pymoo.algorithms.moo.nsga2       import NSGA2
from pymoo.core.problem               import ElementwiseProblem
from pymoo.operators.sampling.rnd     import BinaryRandomSampling
from pymoo.operators.crossover.ux     import UniformCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.optimize                   import minimize as pymoo_minimize

class GraphAnonProblem(ElementwiseProblem):
    def __init__(self, G0, k=2):
        self.G0        = G0
        self.k         = k
        self.all_edges = list(combinations(range(G0.number_of_nodes()), 2))
        super().__init__(n_var=len(self.all_edges), n_obj=2,
                         xl=0, xu=1, vtype=bool)

    def _evaluate(self, x, out, *args, **kwargs):
        G  = apply_flips(self.G0, x)
        rp = compute_k_anonymity_reward(G, self.k)
        ru = compute_utility_reward(G, self.G0)
        out['F'] = [-rp, -ru]   # pymoo minimiza

POP_SIZE = 25
N_GEN    = 75
print(f'Corriendo NSGA-II  pop={POP_SIZE}  gen={N_GEN}  n_var={len(ALL_EDGES)}')

problem      = GraphAnonProblem(G_fb, k=K_ANON)
algorithm    = NSGA2(
    pop_size  = POP_SIZE,
    sampling  = BinaryRandomSampling(),
    crossover = UniformCrossover(),
    mutation  = BitflipMutation(prob=10.0 / len(ALL_EDGES)),
    eliminate_duplicates=True,
)
nsga2_result = pymoo_minimize(
    problem, algorithm,
    termination=('n_gen', N_GEN),
    seed=42, verbose=False,
)

nsga2_pareto = [(-f[0], -f[1]) for f in nsga2_result.F]
results_nsga2 = [(None, p[0], p[1]) for p in nsga2_pareto]
hv_nsga2      = hypervolume_indicator(nsga2_pareto)

# Solucion mas balanceada (max r_priv + r_util)
best_idx = int(np.argmax(-nsga2_result.F[:, 0] + -nsga2_result.F[:, 1]))
G_nsga2  = apply_flips(G_fb, nsga2_result.X[best_idx])

print(f'NSGA-II  HV={hv_nsga2:.5f}  Pareto pts={len(nsga2_pareto)}')

## 7. Baseline: MOEA/D (pymoo)

Descomposicion del problema bi-objetivo en subproblemas escalares con vecindades de peso (Tchebycheff).
Diferencia clave con NSGA-II: en lugar de buscar por dominancia de Pareto, cada subproblema optimiza una
combinacion lineal de objetivos con un vector de pesos fijo — similar a la escalarizacion de MOPPO,
pero sin aprender una politica: busca directamente el grafo final optimo por generacion evolutiva.

**Ventaja conceptual frente a NSGA-II:** puede generar frontes mas uniformes cuando los objetivos
tienen escalas similares.
**Desventaja:** como NSGA-II, debe re-ejecutarse por completo para explorar distintos puntos del frente.

In [ ]:
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.util.ref_dirs        import get_reference_directions

print(f'Corriendo MOEA/D  pop={POP_SIZE}  gen={N_GEN}  n_var={len(ALL_EDGES)}')

# Vectores de referencia uniformes sobre el simplex 2D
ref_dirs = get_reference_directions('uniform', 2, n_partitions=POP_SIZE - 1)

algorithm_moead = MOEAD(
    ref_dirs,
    n_neighbors   = 10,
    decomposition = 'tchebi',  # Tchebycheff scalarization
    prob_neighbor_mating = 0.9,
    sampling  = BinaryRandomSampling(),
    crossover = UniformCrossover(),
    mutation  = BitflipMutation(prob=10.0 / len(ALL_EDGES)),
)

moead_result = pymoo_minimize(
    problem, algorithm_moead,
    termination=('n_gen', N_GEN),
    seed=42, verbose=False,
)

moead_pareto  = [(-f[0], -f[1]) for f in moead_result.F]
results_moead = [(None, p[0], p[1]) for p in moead_pareto]
hv_moead      = hypervolume_indicator(moead_pareto)

# Solucion mas balanceada
best_idx_md = int(np.argmax(-moead_result.F[:, 0] + -moead_result.F[:, 1]))
G_moead     = apply_flips(G_fb, moead_result.X[best_idx_md])

print(f'MOEA/D  HV={hv_moead:.5f}  Pareto pts={len(moead_pareto)}')


## 8. Comparacion Unificada — Pareto Frontier


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

# Aleatorio
ax.scatter([r[1] for r in results_random], [r[2] for r in results_random],
           c='lightgray', s=50, alpha=0.7, label='Aleatorio', zorder=1)

# NSGA-II
nsga2_sorted = sorted(nsga2_pareto, key=lambda p: p[0])
ax.plot([p[0] for p in nsga2_sorted], [p[1] for p in nsga2_sorted],
        's--', color='#43A047', markersize=7, linewidth=1.5,
        label=f'NSGA-II  HV={hv_nsga2:.4f}', zorder=3)


# MOEA/D
moead_sorted = sorted(moead_pareto, key=lambda p: p[0])
ax.plot([p[0] for p in moead_sorted], [p[1] for p in moead_sorted],
        '^--', color='#8E24AA', markersize=7, linewidth=1.5,
        label=f'MOEA/D  HV={hv_moead:.4f}', zorder=3)

# Fixed-alpha
ax.scatter([r[1] for r in results_fixed_all], [r[2] for r in results_fixed_all],
           marker='D', s=130, c='#FF9800', zorder=4,
           label=f'Fixed-alpha PPO K={len(FIXED_ALPHAS)}  HV={hv_fixed:.4f}')
for alpha, rp, ru in results_fixed_all:
    ax.annotate(f'a={alpha:.2f}', (rp, ru), textcoords='offset points',
                xytext=(5, 4), fontsize=7, color='#E65100')

# Greedy
ax.scatter([rp_greedy], [ru_greedy], marker='X', s=320, c='#E53935', zorder=5,
           label=f'Greedy Liu & Terzi  HV={hv_greedy:.4f}')

# MOPPO
moppo_pts    = [(r[1], r[2]) for r in results_moppo]
pareto_idx_m = compute_pareto_front(moppo_pts)
pareto_m     = sorted([moppo_pts[i] for i in pareto_idx_m], key=lambda p: p[0])
ax.scatter([p[0] for p in moppo_pts], [p[1] for p in moppo_pts],
           c='#1E88E5', s=60, zorder=6, alpha=0.4)
ax.plot([p[0] for p in pareto_m], [p[1] for p in pareto_m],
        'o-', color='#1E88E5', markersize=9, linewidth=2.5,
        label=f'MOPPO (ours)  HV={hv_moppo:.4f}  (1 training run)', zorder=7)

ax.set_xlabel('r_priv  (k-anonimato, mayor = mas privado)', fontsize=11)
ax.set_ylabel('r_util  (clustering coeff, mayor = mas util)', fontsize=11)
ax.set_title('Comparacion de Frentes de Pareto — ego-Facebook (n=100)', fontsize=13)
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/pareto_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {OUT_DIR}/pareto_comparison.png')

In [ ]:
n_pareto_m = len(compute_pareto_front(moppo_pts))
n_pareto_f = len(compute_pareto_front([(r[1], r[2]) for r in results_fixed_all]))
n_pareto_n = len(nsga2_pareto)
n_pareto_md = len(moead_pareto)
n_pareto_r = len(compute_pareto_front([(r[1], r[2]) for r in results_random]))

df_hv = pd.DataFrame({
    'Metodo'          : ['MOPPO (ours)', f'Fixed-alpha K={len(FIXED_ALPHAS)}',
                         'NSGA-II', 'MOEA/D', 'Greedy Liu & Terzi', 'Aleatorio'],
    'Hypervolume'     : [hv_moppo, hv_fixed, hv_nsga2, hv_moead, hv_greedy, hv_random],
    'Pareto pts'      : [n_pareto_m, n_pareto_f, n_pareto_n, len(moead_pareto), 1, n_pareto_r],
    'Training runs'   : [1, len(FIXED_ALPHAS), 0, 0, 0, 0],
    'Inf. adaptable'  : ['Si (cambiar alpha = gratis)',
                          'No (un modelo por alpha)',
                          'No (re-correr para c/alpha)',
                          'No (re-correr para c/alpha)',
                          'No (punto fijo)',
                          'No'],
}).set_index('Metodo').sort_values('Hypervolume', ascending=False)

print('Tabla de comparacion (ordenada por Hypervolume):')
display(df_hv)

best_method = df_hv.index[0]
hv_best     = df_hv['Hypervolume'].iloc[0]
hv_moppo_v  = df_hv.loc['MOPPO (ours)', 'Hypervolume']

print(f'\nMejor HV absoluto: {best_method}  ({hv_best:.5f})')
if best_method != 'MOPPO (ours)':
    pct = hv_moppo_v / hv_best * 100
    print(f'MOPPO alcanza {pct:.1f}% del HV maximo con 1 training run.')
    print('Argumento: MOPPO es competitivo pero eficiente y adaptable en inferencia.')
    print(f'{best_method} debe re-ejecutarse para cada nuevo alpha; MOPPO no.')
else:
    print('MOPPO domina en HV con 1 training run — argumento directo de superioridad.')

## 9. Fairness Audit Comparativo


Para cada metodo se corre un episodio balanceado (alpha=0.5) y se mide el disparity index Δ = Q1/Q4.  
**Δ = 1** = equitativo | **Δ > 1.5** = sesgo contra nodos perifericos (Rawls: injusto).

In [ ]:
def audit_model(model, name, alpha=0.5):
    G_final, G0 = get_final_graph(make_env(), model, alpha)
    stats       = degree_quartile_audit(G0, G_final)
    return {'name': name, 'G_final': G_final, 'stats': stats,
            'disparity': fairness_disparity_index(stats)}

def audit_graph_static(G_final, G0, name):
    stats = degree_quartile_audit(G0, G_final)
    return {'name': name, 'G_final': G_final, 'stats': stats,
            'disparity': fairness_disparity_index(stats)}

audits = [
    audit_model(model_moppo,          'MOPPO',           alpha=0.5),
    audit_model(fixed_models[0.5],    'Fixed-alpha 0.5', alpha=0.5),
    audit_graph_static(G_nsga2,  G_fb, 'NSGA-II'),
    audit_graph_static(G_greedy, G_fb, 'Greedy'),
]

rows = []
for a in audits:
    st  = a['stats']
    q1  = st.get('Q1_peripheral', {}).get('normalized_loss', float('nan'))
    mid = st.get('Q2Q3_middle',   {}).get('normalized_loss', float('nan'))
    q4  = st.get('Q4_hub',        {}).get('normalized_loss', float('nan'))
    rows.append({
        'Metodo'        : a['name'],
        'Q1 norm_loss'  : round(q1,  4),
        'Q2Q3 norm_loss': round(mid, 4),
        'Q4 norm_loss'  : round(q4,  4),
        'Disparity Δ'   : round(a['disparity'], 4),
        'Justo (Δ<=1.5)': a['disparity'] <= 1.5,
    })

df_audit = pd.DataFrame(rows).set_index('Metodo').sort_values('Disparity Δ')
print('Fairness Audit Comparativo (menor Δ = mas justo, Δ=1 ideal):')
display(df_audit)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

method_names = [a['name'] for a in audits]
disparities  = [a['disparity'] for a in audits]
bar_colors   = ['#1E88E5' if d <= 1.5 else '#E53935' for d in disparities]

# Panel 1: disparity index
bars = axes[0].bar(method_names, disparities, color=bar_colors, edgecolor='white')
axes[0].axhline(y=1.0, color='green', linestyle='--', linewidth=1.5, label='Δ=1 (ideal)')
axes[0].axhline(y=1.5, color='red',   linestyle='--', linewidth=1.5, label='Δ=1.5 (umbral)')
axes[0].set_ylabel('Disparity index  Δ = Q1 / Q4')
axes[0].set_title('Fairness — Disparity index por metodo\nAzul OK (Δ<=1.5) | Rojo sesgado')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=15)
for bar, val in zip(bars, disparities):
    axes[0].text(bar.get_x() + bar.get_width() / 2.0,
                 bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Panel 2: desglose Q1/Q2Q3/Q4
x     = np.arange(len(audits))
w     = 0.25
q1v   = [a['stats'].get('Q1_peripheral', {}).get('normalized_loss', 0) for a in audits]
midv  = [a['stats'].get('Q2Q3_middle',   {}).get('normalized_loss', 0) for a in audits]
q4v   = [a['stats'].get('Q4_hub',         {}).get('normalized_loss', 0) for a in audits]

axes[1].bar(x - w, q1v,  w, label='Q1 periferia', color='#2196F3')
axes[1].bar(x,     midv, w, label='Q2Q3 medio',   color='#4CAF50')
axes[1].bar(x + w, q4v,  w, label='Q4 hub',       color='#F44336')
axes[1].set_xticks(x)
axes[1].set_xticklabels(method_names, rotation=15)
axes[1].set_ylabel('Perdida de aristas normalizada')
axes[1].set_title('Costo de anonimizacion por cuartil\n(ideal: barras iguales entre grupos)')
axes[1].legend(fontsize=8)

plt.suptitle('Fairness Audit Comparativo — ego-Facebook (n=100)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fairness_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {OUT_DIR}/fairness_comparison.png')

## 10. TREX — Analisis de Trayectorias para Auditoria Etica

Trajectory clustering sobre 30 episodios con alpha aleatorio.
Cada episodio produce un perfil cumulativo [R_priv, R_util] a lo largo de T pasos.
K-means (k=3) agrupa las trayectorias en modos conductuales:
- **utility-dominant** (alpha bajo): el agente preserva la estructura
- **balanced** (alpha medio): balance entre privacidad y utilidad
- **privacy-dominant** (alpha alto): el agente maximiza k-anonimato

**Pregunta etica clave:** ¿Alguno de estos modos impone un costo desproporcionado
sobre nodos perifericos (Q1) vs hubs (Q4)? Si Delta > 1.5 en algun cluster,
ese modo de comportamiento es Rawlsianamente injusto.

In [ ]:
import sys
sys.path.insert(0, '..')
from graph_anon_morl.trex import run_trex_analysis, plot_trex_results

print('Corriendo TREX (30 episodios, k=3)...')
trex_result = run_trex_analysis(
    make_env(), model_moppo,
    n_episodes=30,
    n_clusters=3,
    device=str(device),
    seed=42,
)

fig = plot_trex_results(trex_result, out_path=f'{OUT_DIR}/trex_analysis.png')
plt.show()

print('\nResumen por cluster:')
for cs in trex_result['cluster_stats']:
    fair = 'OK' if cs['mean_disparity'] <= 1.5 else 'SESGO'
    print(f"  [{cs['semantic']:20s}]  "
          f"n={cs['n_episodes']:2d}  "
          f"mean_alpha={cs['mean_alpha']:.2f}  "
          f"Delta={cs['mean_disparity']:.3f} [{fair}]")


In [ ]:
# Tabla resumen para el paper (seccion 6.4)
import pandas as pd

rows = []
for cs in trex_result['cluster_stats']:
    rows.append({
        'Cluster':         cs['semantic'],
        'n epis.':         cs['n_episodes'],
        'mean alpha':      f"{cs['mean_alpha']:.2f} ± {cs['std_alpha']:.2f}",
        'r_priv final':    f"{cs['mean_r_priv']:.3f}",
        'r_util final':    f"{cs['mean_r_util']:.3f}",
        'Delta (Q1/Q4)':   f"{cs['mean_disparity']:.3f} ± {cs['std_disparity']:.3f}",
        'Rawls':           'OK' if cs['mean_disparity'] <= 1.5 else 'SESGO',
    })

df_trex = pd.DataFrame(rows).set_index('Cluster')
print('Tabla TREX para seccion 6.4 del paper:')
display(df_trex)

# Interpretacion automatica
sesgados = [cs['semantic'] for cs in trex_result['cluster_stats'] if cs['mean_disparity'] > 1.5]
if sesgados:
    print(f'\nATENCION: Los clusters {sesgados} tienen Delta > 1.5 (injusticia Rawlsiana).')
    print('Recomendacion: agregar Delta como tercer objetivo o usar fair_weight_penalty.')
else:
    print('\nTodos los clusters satisfacen el criterio de equidad Rawlsiano (Delta <= 1.5).')


In [ ]:
print('=' * 65)
print('RESUMEN EJECUTIVO PARA EL PAPER')
print('=' * 65)

print('\nHypervolume (mayor = mejor cobertura del frente de Pareto):')
for method, row in df_hv.iterrows():
    hv_val = row['Hypervolume']
    runs   = row['Training runs']
    tag    = '  <-- MOPPO' if method == 'MOPPO (ours)' else ''
    print(f'  {method:28s}: {hv_val:.5f}  ({runs} runs){tag}')

print('\nFairness (menor Δ = mas justo, 1.0 = ideal, >1.5 = sesgado):')
for name, row in df_audit.iterrows():
    disp    = row['Disparity Δ']
    verdict = 'OK     ' if disp <= 1.5 else 'SESGADO'
    tag     = '  <-- MOPPO' if name == 'MOPPO' else ''
    print(f'  {name:28s}: Δ={disp:.3f}  [{verdict}]{tag}')

print('\n--- Argumentos para el paper ---')
best_hv_name = df_hv.index[0]
if best_hv_name == 'MOPPO (ours)':
    print('  [HV] MOPPO lidera: superioridad tecnica + eficiencia de entrenamiento.')
else:
    pct = df_hv.loc['MOPPO (ours)', 'Hypervolume'] / df_hv['Hypervolume'].max() * 100
    print(f'  [HV] MOPPO alcanza {pct:.1f}% del HV maximo con 1 training run.')
    print(f'       {best_hv_name} tiene mas HV pero re-corre para cada alpha nuevo.')
    print('       Ventaja de MOPPO: eficiencia + inferencia adaptable sin re-entrenamiento.')

moppo_disp = df_audit.loc['MOPPO', 'Disparity Δ'] if 'MOPPO' in df_audit.index else None
if moppo_disp is not None:
    if moppo_disp <= 1.5:
        print(f'  [FAIR] MOPPO satisface el criterio de equidad (Δ={moppo_disp:.3f} <= 1.5).')
    else:
        print(f'  [FAIR] MOPPO muestra sesgo (Δ={moppo_disp:.3f} > 1.5).')
        print('         Recomendacion: agregar Δ como tercer objetivo o restriccion.')